# Customer Churn Prediction and Retention Analysis
**Deliverable:** `01_data_science_notebook.ipynb`  
**Dataset:** `customer_churn_advanced_dataset.csv` (provided with the assignment)  
**Purpose:** rank customers by churn risk and translate predictions into retention decisions.

> Important: run this notebook from the project folder with the source CSV in the same directory. The final test set is held out from model selection and threshold selection. Results are predictive associations, not proof of causation.

## 1. Setup and reproducibility
Random seed: **42**. The workflow uses a stratified 80/20 train/test split and 3-fold stratified cross-validation. Missing numeric values are imputed with training-fold medians; categorical values use the most frequent training-fold category. Encoding and scaling are fitted inside the pipeline to reduce leakage.

In [ ]:
import sys, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             brier_score_loss, classification_report)
from sklearn.base import clone
from sklearn.inspection import permutation_importance
import joblib

SEED = 42
DATA_PATH = "customer_churn_advanced_dataset.csv"
raw = pd.read_csv(DATA_PATH)
print("Python:", sys.version)
print("Platform:", platform.platform())
print("pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__)
print("Shape:", raw.shape)
display(raw.head())

## 2. Data audit and cleaning decisions
The identifier `customer_id` is retained for traceability but excluded from model features. Rows with missing predictor values are not silently deleted; imputation occurs within the modelling pipeline. Target rows must contain a valid binary value (0 or 1). Inspect the audit output before proceeding.

In [ ]:
print("Data types:\n", raw.dtypes)
print("\\nMissing values:\n", raw.isna().sum().sort_values(ascending=False).head(10))
print("\\nDuplicate full rows:", raw.duplicated().sum())
print("\\nDuplicate customer IDs:", raw["customer_id"].duplicated().sum())
print("\\nTarget counts:\\n", raw["churn"].value_counts(dropna=False))
print("\\nNumeric summary:\\n", raw.select_dtypes(include="number").describe().T)
# Cleaning rule: keep all rows with valid binary target; do not remove outliers automatically.
df = raw.copy()
df["churn"] = pd.to_numeric(df["churn"], errors="coerce")
df = df[df["churn"].isin([0, 1])].copy()
df["churn"] = df["churn"].astype(int)
print("Rows retained after valid-target check:", len(df), "of", len(raw))

## 3. Exploratory data analysis
The following six charts examine target balance and churn patterns across contract, payment method, internet service, tenure, satisfaction, and monthly charges. Read the charts as descriptive patterns; do not infer causality from group differences.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
df["churn"].value_counts().sort_index().rename(index={0:"No churn",1:"Churn"}).plot(kind="bar", ax=axes[0,0], title="Target distribution")
for ax, col, title in zip(axes.flat[1:], ["contract_type","payment_method","internet_service","tenure_months","satisfaction_score"], ["Churn rate by contract","Churn rate by payment method","Churn rate by internet service","Tenure by churn","Satisfaction by churn"]):
    if col in ["contract_type","payment_method","internet_service"]:
        df.groupby(col)["churn"].mean().sort_values(ascending=False).plot(kind="bar", ax=ax, title=title)
        ax.set_ylabel("Churn rate")
    else:
        df.boxplot(column=col, by="churn", ax=ax)
        ax.set_title(title); ax.set_xlabel("Churn (0=no, 1=yes)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,4))
df.boxplot(column="monthly_charges", by="churn")
plt.title("Monthly charges by churn"); plt.suptitle(""); plt.xlabel("Churn (0=no, 1=yes)")
plt.tight_layout(); plt.show()

# Group summaries support interpretation.
for col in ["contract_type","payment_method","internet_service"]:
    print("\\nChurn rate by", col)
    display(df.groupby(col)["churn"].agg(["count","mean"]).sort_values("mean", ascending=False))

## 4. Feature engineering
All engineered predictors are derived from observed input columns, never from `churn`. The features are hypotheses to test, not guaranteed causes: charge per tenure; complaint intensity; payment-risk score; low-satisfaction indicator; support-gap indicator; usage per charge; discounted monthly charge; and short-tenure indicator. Division by zero is converted to missing and imputed within the pipeline.

In [ ]:
def engineer_features(frame):
    x = frame.copy()
    x["charges_per_tenure"] = x["total_charges"] / x["tenure_months"].replace(0, np.nan)
    x["complaint_intensity"] = x["complaints_last_12m"] / 12.0
    x["payment_risk_score"] = x["late_payments_last_12m"] + (x["payment_method"].astype(str).str.lower() == "electronic check").astype(int)
    x["low_satisfaction_flag"] = (x["satisfaction_score"] <= 5).astype(float)
    x["support_gap_flag"] = ((x["support_plan"].astype(str).str.lower() == "no") & (x["tech_support"].astype(str).str.lower() == "no")).astype(int)
    x["usage_per_charge"] = x["monthly_usage_gb"] / x["monthly_charges"].replace(0, np.nan)
    x["discounted_monthly_charge"] = x["monthly_charges"] * (1 - x["discount_pct"] / 100.0)
    x["short_tenure_flag"] = (x["tenure_months"] <= 12).astype(int)
    return x

data = engineer_features(df)
X = data.drop(columns=["customer_id", "churn"], errors="ignore")
y = data["churn"]
numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number", "bool"]).columns.tolist()
print("Numeric features:", len(numeric_features), numeric_features)
print("Categorical features:", len(categorical_features), categorical_features)
display(data.head())

## 5. Train/test split and preprocessing pipeline
A stratified test partition is held out until final evaluation. Numeric imputation/scaling and categorical imputation/one-hot encoding are placed inside a `ColumnTransformer` and `Pipeline`, so preprocessing is learned only from training folds. Class weights are used because the target distribution is skewed; no resampling is applied.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Training churn rate:", round(y_train.mean(), 3), "Test churn rate:", round(y_test.mean(), 3))

## 6. Model development and tuning
Three classifiers are compared: Logistic Regression, Random Forest, and Extra Trees. Grid search selects parameters using mean cross-validated **average precision (PR-AUC)**, useful when positive cases are a priority. The parameter grids are intentionally modest for reproducible execution on a laptop.

In [ ]:
candidates = {
    "Logistic Regression": (Pipeline([("preprocessor", preprocessor), ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))]), {"model__C":[0.1, 1.0]}),
    "Random Forest": (Pipeline([("preprocessor", preprocessor), ("model", RandomForestClassifier(class_weight="balanced", random_state=SEED, n_jobs=1))]), {"model__n_estimators":[100], "model__max_depth":[None,10], "model__min_samples_leaf":[1,3]}),
    "Extra Trees": (Pipeline([("preprocessor", preprocessor), ("model", ExtraTreesClassifier(class_weight="balanced", random_state=SEED, n_jobs=1))]), {"model__n_estimators":[100], "model__max_depth":[None,10], "model__min_samples_leaf":[1,3]})
}
searches = {}
for name, (pipe, grid) in candidates.items():
    search = GridSearchCV(pipe, grid, scoring="average_precision", cv=cv, n_jobs=1, refit=True)
    search.fit(X_train, y_train)
    searches[name] = search
    print(name, "best CV PR-AUC:", round(search.best_score_, 4), "params:", search.best_params_)
best_name = max(searches, key=lambda name: searches[name].best_score_)
best_model = searches[best_name].best_estimator_
print("Selected by CV PR-AUC:", best_name)

## 7. Threshold selection and final test evaluation
A 0.50 threshold is not automatically assumed. The operating threshold is chosen by maximizing F2 on out-of-fold predictions from the training partition, giving recall greater weight than precision under the assumption that missing a potential churner is costly. The test set is used only after model/threshold selection. Accuracy alone is insufficient because it does not show how well churners are found or how many contacted customers would be false positives.

In [ ]:
oof_prob = cross_val_predict(clone(best_model), X_train, y_train, cv=cv, method="predict_proba", n_jobs=1)[:,1]
threshold_grid = np.linspace(0.15, 0.85, 71)
threshold_rows = []
for t in threshold_grid:
    p = (oof_prob >= t).astype(int)
    precision = precision_score(y_train, p, zero_division=0)
    recall = recall_score(y_train, p, zero_division=0)
    f1 = f1_score(y_train, p, zero_division=0)
    f2 = 5*precision*recall/(4*precision+recall) if 4*precision+recall else 0
    threshold_rows.append([t, precision, recall, f1, f2])
threshold_df = pd.DataFrame(threshold_rows, columns=["threshold","precision","recall","f1","f2"])
selected_threshold = float(threshold_df.loc[threshold_df.f2.idxmax(), "threshold"])
display(threshold_df.iloc[(threshold_df.threshold-selected_threshold).abs().argsort()[:5]].sort_values("threshold"))
threshold_df.set_index("threshold")[["precision","recall","f1"]].plot(figsize=(8,4), title="Precision/recall/F1 across thresholds")
plt.show()

rows, test_probabilities = [], {}
for name, search in searches.items():
    probability = search.best_estimator_.predict_proba(X_test)[:,1]
    test_probabilities[name] = probability
    threshold = selected_threshold if name == best_name else 0.50
    prediction = (probability >= threshold).astype(int)
    rows.append({
        "Model": name, "CV PR-AUC": search.best_score_,
        "Precision": precision_score(y_test,prediction,zero_division=0),
        "Recall": recall_score(y_test,prediction,zero_division=0),
        "F1": f1_score(y_test,prediction,zero_division=0),
        "ROC-AUC": roc_auc_score(y_test,probability),
        "PR-AUC": average_precision_score(y_test,probability),
        "Accuracy": accuracy_score(y_test,prediction), "Threshold": threshold
    })
comparison = pd.DataFrame(rows).sort_values("PR-AUC", ascending=False)
display(comparison)
best_probability = test_probabilities[best_name]
best_prediction = (best_probability >= selected_threshold).astype(int)
print("Selected threshold:", round(selected_threshold,3))
print("Confusion matrix [rows actual 0/1, columns predicted 0/1]:\\n", confusion_matrix(y_test,best_prediction))
print(classification_report(y_test,best_prediction,zero_division=0))
print("Brier score:", round(brier_score_loss(y_test,best_probability),4))

## 8. Calibration and robustness checks
The Brier score summarizes probability error (lower is better). The calibration curve below compares predicted probabilities with observed churn frequency. A three-fold CV score is reported above; fold-to-fold variability can be inspected from `search.cv_results_`. Probability calibration is especially important before using probabilities for financial decisions.

In [ ]:
from sklearn.calibration import calibration_curve
fraction_positive, mean_predicted = calibration_curve(y_test, best_probability, n_bins=8, strategy="quantile")
plt.figure(figsize=(5,5))
plt.plot(mean_predicted, fraction_positive, marker="o", label="Model")
plt.plot([0,1],[0,1], linestyle="--", label="Perfect calibration")
plt.xlabel("Mean predicted churn probability"); plt.ylabel("Observed churn fraction")
plt.title("Calibration curve"); plt.legend(); plt.show()

# Robustness: show CV score variation across folds for the selected grid-search configuration.
cv_results = searches[best_name].cv_results_
best_idx = searches[best_name].best_index_
fold_scores = [cv_results[f"split{i}_test_score"][best_idx] for i in range(cv.get_n_splits())]
print("Selected model CV PR-AUC fold scores:", [round(v,4) for v in fold_scores])
print("Mean:", round(float(np.mean(fold_scores)),4), "Std:", round(float(np.std(fold_scores)),4))

## 9. Explainability
Permutation importance measures how much the chosen metric changes when one raw input feature is shuffled. It is a global predictive-importance method, not a causal explanation. Individual profiles below are illustrative high-risk cases; observed attributes are shown to support review, not to claim that any single attribute caused churn.

In [ ]:
importance = permutation_importance(best_model, X_test, y_test, n_repeats=5, random_state=SEED, scoring="average_precision", n_jobs=1)
importance_df = pd.DataFrame({"feature":X.columns, "importance_mean":importance.importances_mean, "importance_std":importance.importances_std}).sort_values("importance_mean", ascending=False)
display(importance_df.head(15))
importance_df.head(12).sort_values("importance_mean").plot(x="feature", y="importance_mean", kind="barh", legend=False, figsize=(8,5), title="Permutation importance (PR-AUC)")
plt.show()

profiles = X_test.copy()
profiles["actual_churn"] = y_test
profiles["predicted_probability"] = best_probability
profiles["predicted_churn"] = best_prediction
high_risk = profiles.sort_values("predicted_probability", ascending=False).head(3)
display(high_risk[["predicted_probability","actual_churn","contract_type","tenure_months","satisfaction_score","complaints_last_12m","late_payments_last_12m","support_plan","tech_support"]])

## 10. Retention intervention simulation
Assumption: the service team can contact only the top 20% of customers in the held-out test sample, ranked by predicted probability. Compare captured churners against the random-contact expectation. Illustrative financial assumptions (not provided by the dataset): contact cost = $5 per customer; value of a retained customer = $100; successful retention among contacted churners = 25%. Replace these assumptions with business-approved values before operational use.

In [ ]:
n_contact = max(1, int(np.ceil(0.20 * len(X_test))))
top_idx = np.argsort(best_probability)[::-1][:n_contact]
captured_churners = int(y_test.iloc[top_idx].sum())
actual_churners = int(y_test.sum())
capture_rate = captured_churners / actual_churners if actual_churners else 0
random_expected_captured = 0.20 * actual_churners

contact_cost = 5.0
retained_customer_value = 100.0
retention_success_rate = 0.25
target_cost = n_contact * contact_cost
target_expected_retained = captured_churners * retention_success_rate
target_expected_value = target_expected_retained * retained_customer_value
target_net = target_expected_value - target_cost
random_cost = n_contact * contact_cost
random_expected_retained = random_expected_captured * retention_success_rate
random_expected_value = random_expected_retained * retained_customer_value
random_net = random_expected_value - random_cost

print(f"Test customers: {len(X_test)}; contact capacity (20%): {n_contact}")
print(f"Actual churners in test set: {actual_churners}")
print(f"Top-risk group captured {captured_churners} churners ({capture_rate:.1%} of all test churners).")
print(f"Random-contact expectation: {random_expected_captured:.1f} churners.")
print(f"Targeted net value: ${target_net:,.2f}; random net value: ${random_net:,.2f}.")
print("Financial figures are hypothetical and depend on the stated assumptions.")

## 11. Save deliverables and conclusions
The model bundle contains the fitted pipeline, selected threshold, feature list, model name, and seed. The cleaned CSV retains the customer identifier and target for traceability; the identifier is not included in `X`. Use the model on future customers only when the input schema matches the features used here.

In [ ]:
data.to_csv("02_cleaned_dataset.csv", index=False)
joblib.dump({"pipeline":best_model, "threshold":selected_threshold, "feature_columns":X.columns.tolist(), "model_name":best_name, "random_state":SEED}, "03_model_artifact.joblib")
comparison.to_csv("model_comparison.csv", index=False)
importance_df.to_csv("feature_importance.csv", index=False)
print("Saved cleaned dataset, model artifact, model comparison, and feature importance.")
print("Selected model:", best_name)
print("Final test metrics:", {
    "precision": precision_score(y_test,best_prediction,zero_division=0),
    "recall": recall_score(y_test,best_prediction,zero_division=0),
    "F1": f1_score(y_test,best_prediction,zero_division=0),
    "ROC-AUC": roc_auc_score(y_test,best_probability),
    "PR-AUC": average_precision_score(y_test,best_probability),
    "accuracy": accuracy_score(y_test,best_prediction)
})